# ***Projeto Teste A/B***

***Objetivo***: *verificar se a nova página de um produto tem conversão de 15% (a página atual tem conversão de 13%) 
e quais seriam os potenciais números de vendas e faturamentos advindos dessa nova página.*

## 1.0 Imports

### 1.1 Bibliotecas:

In [ ]:
import pandas as pd
import numpy as np
import math
import seaborn as sns
from matplotlib import pyplot as plt
from statsmodels.stats import api as sms
from scipy.stats import chi2_contingency

### 1.2 Carregando os dados:

In [ ]:
df_raw = pd.read_csv('../datasets/ab_data.csv', low_memory=False)

In [ ]:
df_raw

## 2.0 Design do experimento

### 2.1 Formulação das hipóteses

In [ ]:
# Hipóteses:
# H0: a conversão da nova página é de 13%
# H1: a conversão da nova página é diferente de 13%

### 2.2 Definição dos parâmetros do teste

In [ ]:
# Nível de confiança:
# é a probabilidade de que um intervalo estimado (amostra) contenha o verdadeiro parâmetro populacional
# 90%, 95% e 99% são valores comumente adotados
nivel_confianca = 0.95

In [ ]:
# Nível de significância:
# é probabilidade de rejeitar a hipótese nula quando ela é verdadeira
# definida como (1 - nível de confiança)
nivel_significancia = 0.05

In [ ]:
# Tamanho do efeito (effect size):
# é uma medida quantitativa que indica a força da relação entre duas variáveis ou a intensidade da diferença entre grupos
# também pode ser entendida como quão grande é a diferença que esperamos que haja entre as variáveis analisadas
# uma maneira de determinar é através do d de Cohen: dividindo a diferença entre as duas médias populacionais pelo seu desvio padrão

# obs: nesse caso, não temos as médias e sim as proporções

# conversões das páginas:
p1 = 0.13
p2 = 0.15

effect_size = sms.proportion_effectsize(p1, p2)    # esse cálculo é baseado em h de Cohen

In [ ]:
# Poder estatístico:
# é a probabilidade em que um teste pode detectar corretamente um efeito real quando realmente existe um
# comumente fixada em 80%
power = 0.80

In [ ]:
# Em suma, o effect size informa a relevância prática, enquanto o poder estatístico informa a 
# probabilidade de identificar essa relevância.

In [ ]:
# Tamanho da amostra:
# tamanho mínimo que cada grupo da amostra precisa ter para atender aos parâmetros acima
amostra = sms.NormalIndPower().solve_power(effect_size = effect_size,
                                          power = power,
                                          alpha = nivel_significancia)
amostra = math.ceil(amostra)
print('Tamanho mínimo para cada grupo da amostra: {}'.format(amostra))

## 3.0 Descrição dos dados

In [ ]:
df = df_raw.copy()

In [ ]:
df.info()

In [ ]:
# mudando o dtype da coluna 'timestamp':
df['timestamp'] = pd.to_datetime(df['timestamp']).apply(lambda x: x.strftime('%Y-%m-%d'))

In [ ]:
# verificando se há linhas duplicadas:
linhas_duplicadas = df[df.duplicated()]
print("Linhas duplicadas:")
print(linhas_duplicadas)

In [ ]:
# conferindo as "flags" do teste ab:
df[['user_id', 'group', 'landing_page']].groupby(['group', 'landing_page']).count().reset_index()

In [ ]:
# o que seria esperado para as flags do teste ab seria algo do tipo (exemplo):

# group/landing     old_page    new_page
# control            1000          0
# treatment            0          1000

In [ ]:
# deletando 'user_id' que aparecem em mais de um grupo:
df_user_delete = df[['user_id', 'group']].groupby('user_id').count().reset_index().query('group > 1')['user_id']
df1 = df[~df['user_id'].isin(df_user_delete)]

df1[['user_id', 'group', 'landing_page']].groupby(['group', 'landing_page']).count().reset_index()

In [ ]:
# amostragem aleatória dos grupos controle e tratamento:
df_control_sample = df1[df1['group'] == 'control'].sample(n=amostra, random_state=42)
df_treatment_sample = df1[df1['group'] == 'treatment'].sample(n=amostra, random_state=42)

#concatenando os grupos criados acima:
df_ab = pd.concat([df_control_sample, df_treatment_sample]).reset_index(drop=True)

In [ ]:
# calculando a taxa de conversão dos grupos (métrica de interesse):
# aqui, conversão = compradores / visitantes

# ==================== Grupo controle ============================
compradores = df_control_sample.loc[df_control_sample ['converted'] == 1, 'converted'].sum()
visitantes = len(df_control_sample)

taxa_conversao = compradores/visitantes
print('Taxa de conversão - Grupo controle: {:.4f}'.format(taxa_conversao))

# ==================== Grupo tratamento ==========================
compradores = df_treatment_sample.loc[df_treatment_sample ['converted'] == 1, 'converted'].sum()
visitantes = len(df_treatment_sample)

taxa_conversao = compradores/visitantes
print('Taxa de conversão - Grupo tratamento: {:.4f}'.format(taxa_conversao))

In [ ]:
### GRÁFICO: Comparação de Taxas de Conversão

groups = ['Página Atual\n(Controle)', 'Página Nova\n(Tratamento)']
conversoes = [11.55, 12.90]
target = [15, 15]

fig, ax = plt.subplots(figsize=(8, 4))

# Barras de conversão
bars = ax.bar(groups, conversoes, color=['#3498db', '#e74c3c'], alpha=0.7, label='Conversão Observada')

# Linha de meta
ax.axhline(y=15, color='green', linestyle='--', linewidth=2, label='Meta (15%)')
ax.axhline(y=13, color='orange', linestyle='--', linewidth=2, label='Baseline (13%)')

# Adicionando valores nas barras
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{conversoes[i]:.2f}%',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Taxa de Conversão (%)', fontsize=12)
ax.set_title('Teste A/B: Comparação de Conversão entre Páginas', fontsize=14)
ax.set_ylim(0, 17)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/conversion_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### GRÁFICO: Funil de Conversão

# Dados do funil
stages = ['Visitantes', 'Conversões']
controle = [4720, 545]
tratamento = [4720, 609]

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Funil Controle
ax[0].barh(stages, controle, color='#3498db', alpha=0.7)
ax[0].set_title('Página Atual (Controle)\nConversão: 11.55%', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Número de Usuários', fontsize=10)
for i, v in enumerate(controle):
    ax[0].text(v + 100, i, str(v), va='center', fontsize=11, fontweight='bold')

# Funil Tratamento
ax[1].barh(stages, tratamento, color='#e74c3c', alpha=0.7)
ax[1].set_title('Página Nova (Tratamento)\nConversão: 12.90%', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Número de Usuários', fontsize=10)
for i, v in enumerate(tratamento):
    ax[1].text(v + 100, i, str(v), va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/conversion_funnel.png', dpi=300, bbox_inches='tight')
plt.show()

## 4.0 Inferência estatística

In [ ]:
# razão da escolha do teste estatístico:
# o teste Qui-quadrado é usado para dados categóricos (frequências) para verificar associações ou aderência
# permite testar se os dados observados seguem uma distribuição esperada
# requer amostras com n > 5 
# mede a diferença entre frequências observadas e esperadas (tabelas de contingência)

In [ ]:
# testando as hipóteses:
# inicialmente, 'transformando' a tabela concatenada para que possa ser lida pelo chi2_contingency: 
df_table = df_ab[['group', 'converted']].groupby('group').agg({'converted':['sum', 'count']})
df_table.columns = ['converted', 'total']

# os resultados de chi2_contingency:
chi_val, pval, dof, freq_expected = chi2_contingency(df_table)

# comparando o p-valor com o nível de significância:
if pval < nivel_significancia:
    print('Rejeita a hipótese nula')
    print(f'RESULTADO SIGNIFICATIVO (p-value: {pval:.4f})')
    print('A nova página tem conversão estatisticamente superior.')
else:
    print('Não rejeita a hipótese nula')
    print(f'SEM SIGNIFICÂNCIA (p-value: {pval:.4f})')
    print('Não há evidência estatística suficiente de que a nova página é melhor.')

In [ ]:
### GRÁFICO: Resultados do Teste Estatístico

# Visualização do p-valor
fig, ax = plt.subplots(figsize=(8, 4))

categories = ['Controle vs Tratamento']
p_value = [0.0806]  # p-valor medido

colors = ['red' if p > 0.05 else 'green' for p in p_value]
bars = ax.bar(categories, p_value, color=colors, alpha=0.6)

# Linha de significância
ax.axhline(y=0.05, color='black', linestyle='--', linewidth=2, label='Nível de Significância (α=0.05)')

# Adicionando p-valor nas barras
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'p = {p_value[i]:.3f}\n(Não significativo)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('P-valor', fontsize=12)
ax.set_title('Resultado do Teste Chi-Squared\nP-valor > 0.05 = Não há diferença significativa', fontsize=12)
ax.set_ylim(0, 0.5)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/statistical_test.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.1 Conversão da página em faturamento

In [ ]:
# acima, vimos que o resultado para p-valor não permite que rejeitemos a hipótese nula
# mas, vamos supor que tivéssemos obtido "Rejeita a hipótese nula"

In [ ]:
df2 = df1.copy()

In [ ]:
# lembrando, conversão da página atual = 13% e conversão da página nova = 15%

# compradores = número de visitantes diário * 0.13
# faturamento = 4500 * compradores

In [ ]:
df3 = df2[['timestamp', 'user_id']].groupby('timestamp').count().reset_index()

# Faturamento atual:
df3['compradores_atual'] = np.round(df3['user_id'] * 0.13).astype(int)
df3['fat_atual'] = df3['compradores_atual'] * 4500

fat_atual = df3['fat_atual'].sum()
print('Faturamento atual no período: "R${:,.2f}"'.format(fat_atual))

# Faturamento esperado:
df3['compradores_esp'] = np.round(df3['user_id'] * 0.15).astype(int)
df3['fat_esp'] = df3['compradores_esp'] * 4500

fat_esp = df3['fat_esp'].sum()
print('Faturamento esperado no período: "R${:,.2f}"'.format(fat_esp))

# Cálculo do aumento do faturamento:
lift = 100 * (fat_esp - fat_atual) / fat_atual
print('Lift esperado: {:.2f}%'.format(lift))

In [ ]:
# Com esses valores de faturamento, o número de unidades do produto que foram vendidas é:
# atual:
un_vend_atual = (fat_atual / 4500).astype(int)
print('Unidades vendidas atual: {}'.format(un_vend_atual))

# esperado:
un_vend_esp = (fat_esp / 4500).astype(int)
print('Unidades vendidas esperado: {}'.format(un_vend_esp))

# diferença entre atual e esperado:
aum = un_vend_esp - un_vend_atual
print('O aumento de unidades vendidas é de: {}'.format(aum))

In [ ]:
### GRÁFICO: Potencial de Receita

# Cenários de faturamento
cenarios = ['Atual\n(11.55%)', 'Observado\nNova Página\n(12.90%)', 'Baseline\n(13%)', 'Meta\n(15%)']
revenue = [(fat_atual*0.8885)/1e6, (fat_atual*0.9923)/1e6, fat_atual/1e6, fat_esp/1e6]  # em milhões

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(cenarios, revenue, color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'], alpha=0.7)

# Adicionando valores
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'R$ {revenue[i]:.2f}M', ha='center', va='bottom', fontsize=12)

ax.set_ylabel('Faturamento (R$ Milhões)', fontsize=12)
ax.set_title('Comparação de Faturamento Potencial\nProduto: Teclado Bluetooth (R$ 4.500)', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/revenue_potential.png', dpi=300, bbox_inches='tight')
plt.show()

## 5.0 Resultados

RESULTADOS PRINCIPAIS:
- Taxa de conversão grupo CONTROLE: 11.55%
- Taxa de conversão grupo TRATAMENTO: 12.90%
- Diferença: 1.35 pontos percentuais
- P-valor: 0.0806 (não significativo)
- Conclusão: Nova página NÃO é melhor que a atual

##### MÉTRICAS PRINCIPAIS:

**Dataset:**
- 294.478 usuários totais
- Após limpeza: 286.690 usuários válidos
- Amostra por grupo: 4.720 usuários (9.440 total)

**Conversões Observadas:**
- Controle (página atual): 11.55% (545/4720 conversões)
- Tratamento (página nova): 12.90% (609/4720 conversões)
- Diferença: 1.35 pontos percentuais

**Teste Estatístico:**
- Método: Chi-Squared Test
- P-valor: 0.0806 (acima de 0.05)
- **Conclusão:** NÃO há diferença estatisticamente significativa

**Implicação de Negócio:**
- Meta era 15% (2% de aumento sobre 13%)
- Resultado real: 12.90% (ABAIXO da baseline)
- **Recomendação:** NÃO implementar a nova página

**Impacto Financeiro SE a meta fosse atingida:**
- Vendas atuais estimadas: 37.271 unidades
- Vendas esperadas (15%): 43.004 unidades
- Aumento potencial: 5.733 unidades
- Receita adicional potencial: R$ 25.798.500,00

## 6.0 Insights e recomendações

### 6.1 Principais Insights

#### 1. Resultado do Teste
- A nova página NÃO apresentou melhoria na conversão
- Conversão observada (12.90%) está ABAIXO da baseline (13%)
- P-valor de 0.0806 indica que a diferença não é estatisticamente significativa

#### 2. Distância da Meta
- Meta estabelecida: 15% (aumento de 2 pontos percentuais)
- Resultado observado: 12.90%
- Gap: 2.10 pontos percentuais abaixo da meta

#### 3. Impacto Financeiro
Se a meta de 15% fosse atingida:
- Aumento de 5.733 unidades vendidas
- Receita adicional: R$ 25.798.500,00
- Lift: 15.38% no faturamento

#### 4. Qualidade do Experimento
- Amostra suficiente (4.720 por grupo)
- Randomização adequada
- Limpeza de dados rigorosa (flags incorretas removidas)

### 6.2 Recomendações

- **NÃO implementar a nova página**
- Realizar sessões de feedback com usuários
- Investigar pontos problemáticos na jornada dos usuários
- Redesign da nova página baseado em pesquisa qualitativa
- Novo teste A/B com versão melhorada
- Testar elementos individuais (preço, imagens, CTA)

## 7.0 Conclusão

### Pergunta 1: A conversão da nova página é realmente melhor que a página atual?

**Resposta: NÃO**

- Taxa de conversão observada: 12.90% (tratamento) vs 11.55% (controle)
- Diferença: 1.35 pontos percentuais (favorece página antiga quando considerado o baseline 13%)
- Teste estatístico: p-valor = 0.0806 (não significativo)
- **Conclusão:** Não há evidência de que a nova página é melhor. As páginas são estatisticamente equivalentes.

### Pergunta 2: Qual o potencial de número de vendas que a nova página pode trazer?

**Resposta: Com a conversão observada (12.90%), MENOS vendas que o esperado**

Cenário Real (Conversão = 11.55%):
- Vendas estimadas: 33.113 unidades
- vs Baseline (13%): 37.271 unidades
- Diferença: -4.158 unidades (-11.16%)

Cenário Ideal (Se meta de 15% fosse atingida):
- Vendas potenciais: 43.004 unidades
- vs Baseline: +5.733 unidades (+15.38%)

### Pergunta 3: Qual o faturamento total na venda do teclado bluetooth através da nova página?

**Resposta: Com a conversão observada, MENOR faturamento**

Cenário Real (11.55%):
- Faturamento estimado: R\$149.008.500,00
- comparação com Baseline (13%): R\$167.719.500,00
- Diferença: -R\$18.711.000,00 (-11.16%)

Cenário Ideal (Se meta de 15% fosse atingida):
- Faturamento potencial: R\$193.518.000,00
- comparação com Baseline: +R\$25.798.500,00 (+15.38%)
- Lift: 15.38%

**RECOMENDAÇÃO: NÃO IMPLEMENTAR A NOVA PÁGINA**

**Justificativas:**
1. Conversão inferior ao baseline (12.90% < 13.00%)
2. Meta de 15% não atingida (gap de 2.10 pp)
3. Sem significância estatística (p-valor = 0.0806)
4. Risco de perda de R$1.29M em faturamento em relação ao baseline